# ДЗ 1, исследовательская часть

Анализ данных и замеры скорости.

Само приложение в `app.py`, общие функции в `utils.py`.

In [1]:
import os, time
import numpy as np
import pandas as pd

import utils

df = utils.load_data("temperature_data.csv")
print(df.shape)
df.head()

(54750, 4)


,city,timestamp,temperature,season
0,Beijing,2010-01-01,-1.025049,winter
1,Beijing,2010-01-02,-6.295520,winter
2,Beijing,2010-01-03,-5.294014,winter
3,Beijing,2010-01-04,0.267899,winter
4,Beijing,2010-01-05,2.667202,winter


## 1. Анализ исторических данных

Скользящее среднее за 30 дней, средние и std по сезонам, аномалии.

In [2]:
data = utils.analyze_vectorized(df)
data.head()

,city,timestamp,temperature,season,roll_mean,roll_std,anomaly_roll,season_mean,season_std,anomaly_season
0,Beijing,2010-01-01,-1.025049,winter,-1.025049,NaN,False,-1.951337,4.907936,False
1,Beijing,2010-01-02,-6.295520,winter,-3.660284,3.726786,False,-1.951337,4.907936,False
2,Beijing,2010-01-03,-5.294014,winter,-4.204861,2.798957,False,-1.951337,4.907936,False
3,Beijing,2010-01-04,0.267899,winter,-3.086671,3.197525,False,-1.951337,4.907936,False
4,Beijing,2010-01-05,2.667202,winter,-1.935896,3.780150,False,-1.951337,4.907936,False


In [3]:
seasonal = (data.groupby(["city", "season"])["temperature"]
                .agg(mean="mean", std="std")
                .reset_index())
seasonal.head(8)

,city,season,mean,std
0,Beijing,autumn,15.793107,4.941333
1,Beijing,spring,13.187717,5.091534
2,Beijing,summer,26.806063,5.079494
3,Beijing,winter,-1.951337,4.907936
4,Berlin,autumn,11.212017,4.846098
5,Berlin,spring,10.044254,4.911238
6,Berlin,summer,19.635637,4.901918
7,Berlin,winter,-0.383679,4.683119


### Аномалии

Считаю двумя способами:

* `anomaly_season`: отклонение от средней по сезону больше 2 сигм
* `anomaly_roll`: выход за скользящее среднее плюс-минус 2 скользящих сигмы

Основной первый, с ним потом сравнивается текущая погода из API.

Температуры сгенерированы нормальным распределением, значит аномалий должно быть около 4.6%. Заодно проверка, что я нигде не накосячил.

In [4]:
print("аномалий по сезонной норме: %.2f%%" % (100 * data["anomaly_season"].mean()))
print("аномалий по скользящему окну: %.2f%%" % (100 * data["anomaly_roll"].mean()))
print("ожидаемые 4.55%")

data.groupby("city")[["anomaly_season", "anomaly_roll"]].sum().sort_values("anomaly_season", ascending=False)

аномалий по сезонной норме: 4.41%
аномалий по скользящему окну: 5.10%
ожидаемые 4.55%


,anomaly_season,anomaly_roll
city,,
Cairo,174,215
London,173,166
Tokyo,170,208
Rio de Janeiro,170,166
Singapore,164,131
Mumbai,162,165
Moscow,161,228
New York,161,227
Los Angeles,159,166


### Тренды

Линейная регрессия температуры по времени, наклон перевожу в градусы за год.

In [5]:
summary = pd.DataFrame([utils.city_summary(g) for _, g in data.groupby("city")])
summary.sort_values("trend_C_per_year", ascending=False).round(3)

,city,mean,min,max,std,anomalies,anomalies_%,trend_C_per_year
5,Los Angeles,19.551,0.862,41.401,6.213,159,4.356,0.106
7,Moscow,5.458,-26.252,33.357,11.163,161,4.411,0.102
14,Tokyo,16.635,-10.648,41.848,8.952,170,4.658,0.085
9,New York,12.687,-14.956,44.264,10.254,161,4.411,0.084
1,Berlin,10.182,-14.410,34.994,8.586,151,4.137,0.076
4,London,11.349,-10.884,32.677,6.863,173,4.740,0.061
13,Sydney,18.797,-2.558,44.712,6.888,158,4.329,0.055
2,Cairo,25.125,-1.233,52.009,8.674,174,4.767,0.048
3,Dubai,29.972,3.894,55.098,8.705,153,4.192,0.031
0,Beijing,13.537,-17.047,43.773,11.397,152,4.164,0.023


Тренды почти нулевые, так и должно быть: сезонные средние одинаковые все 10 лет, никакого потепления в данные не заложено.

## 2. Параллельность

1. цикл по городам
2. города по потокам
3. города по процессам
4. без цикла, один `groupby().transform()` на весь датафрейм

In [6]:
def bench(fn, *a, n=5, **kw):
    ts = []
    for _ in range(n):
        t = time.perf_counter()
        fn(*a, **kw)
        ts.append(time.perf_counter() - t)
    return min(ts)  # беру лучшее время, чтобы фоновые процессы не мешали


res = {
    "последовательно (цикл)": bench(utils.analyze_sequential, df),
    "потоки (4)": bench(utils.analyze_parallel, df, 4, "thread"),
    "процессы (4)": bench(utils.analyze_parallel, df, 4, "process", n=3),
    "векторизованно": bench(utils.analyze_vectorized, df),
}
base = res["последовательно (цикл)"]
pd.DataFrame({"сек": res, "ускорение": {k: base / v for k, v in res.items()}}).round(3)

,сек,ускорение
последовательно (цикл),0.023,1.000
потоки (4),0.030,0.793
процессы (4),0.027,0.873
векторизованно,0.008,2.810


Проверим, что все варианты дают одно и то же, вдруг я где-то напутал с сортировкой.

In [7]:
a = utils.analyze_sequential(df).sort_values(["city", "timestamp"]).reset_index(drop=True)
b = utils.analyze_parallel(df, 4, "process").sort_values(["city", "timestamp"]).reset_index(drop=True)
c = utils.analyze_vectorized(df).sort_values(["city", "timestamp"]).reset_index(drop=True)
print("seq == process:", a["anomaly_season"].equals(b["anomaly_season"]))
print("seq == vectorized:", np.allclose(a["roll_mean"], c["roll_mean"]),
      a["anomaly_season"].equals(c["anomaly_season"]))

seq == process: True
seq == vectorized: True True


### Что если данных будет больше

На 55 тысячах строк процессы проигрывают: запустить их и переслать туда данные дороже, чем посчитать.
Увеличим объем данных в 20 раз и посмотрю, изменится ли картина.

In [8]:
big = pd.concat(
    [df.assign(city=df["city"] + f"_{i}") for i in range(20)], ignore_index=True
)
print("строк:", len(big), "| городов:", big["city"].nunique())

res_big = {
    "последовательно (цикл)": bench(utils.analyze_sequential, big, n=3),
    "потоки (8)": bench(utils.analyze_parallel, big, 8, "thread", n=3),
    "процессы (8)": bench(utils.analyze_parallel, big, 8, "process", n=3),
    "векторизованно": bench(utils.analyze_vectorized, big, n=3),
}
base_big = res_big["последовательно (цикл)"]
pd.DataFrame(
    {"сек": res_big, "ускорение": {k: base_big / v for k, v in res_big.items()}}
).round(3)

строк: 1095000 | городов: 300


,сек,ускорение
последовательно (цикл),0.466,1.000
потоки (8),0.613,0.761
процессы (8),0.223,2.094
векторизованно,0.135,3.443


### Выводы по параллельности

На исходных данных параллелить смысла нет, весь анализ считается за 0.02 секунды, а поднять пул процессов
и перекинуть в него датафреймы это долго.

Потоки не помогают нигде: работа упирается в процессор, а не в ожидание, и GIL держится почти все время.

На раздутом в 20 раз датасете процессы наконец выигрывают, примерно в 2 раза. До восьмикратного ускорения
по числу воркеров далеко, заметную часть съедает сериализация кусков датафрейма туда и обратно.

Но даже там быстрее всех оказался вариант вообще без параллелизма, один `groupby().transform()`. Pandas
считает это внутри в C и ничего никуда не копирует, поэтому в приложении используется он.

Процессы тут окупились бы, если бы на каждый город считалось что-то реально тяжелое, вроде подбора SARIMA.

## 3. Синхронные и асинхронные запросы

Ключ берется из переменной окружения `OWM_API_KEY`.

In [9]:
API_KEY = os.environ.get("OWM_API_KEY", "nokey")
cities = sorted(df["city"].unique())
print("городов:", len(cities))

городов: 15


In [10]:
# один город
t = time.perf_counter(); utils.get_weather_sync("Moscow", API_KEY);  t_sync_1 = time.perf_counter() - t
t = time.perf_counter(); utils.get_weather_async("Moscow", API_KEY); t_async_1 = time.perf_counter() - t
print(f"sync:  {t_sync_1:.3f} сек")
print(f"async: {t_async_1:.3f} сек")

sync:  0.315 сек
async: 0.253 сек


In [11]:
# все 15 городов
t = time.perf_counter(); utils.get_weather_sync_many(cities, API_KEY);          t_sync = time.perf_counter() - t
t = time.perf_counter(); utils.get_weather_threads_many(cities, API_KEY, 8);    t_thr  = time.perf_counter() - t
t = time.perf_counter(); utils.get_weather_async_many_blocking(cities, API_KEY); t_async = time.perf_counter() - t

pd.DataFrame({
    "сек": {"sync (requests по очереди)": t_sync, "потоки (8)": t_thr, "async (aiohttp.gather)": t_async},
    "ускорение": {"sync (requests по очереди)": 1, "потоки (8)": t_sync / t_thr, "async (aiohttp.gather)": t_sync / t_async},
}).round(3)

,сек,ускорение
sync (requests по очереди),1.214,1.000
потоки (8),0.707,1.717
async (aiohttp.gather),0.263,4.620


### Выводы по sync и async

Запросы к API это ожидание сети, процессор при этом простаивает.

Для одного города разницы никакой, async даже чуть медленнее, потому что платит за создание event loop.
В приложении запрос ровно один, поэтому там по умолчанию обычный requests: короче код и не надо
запускать asyncio.run внутри синхронного streamlit.

Для неск. городов async быстрее в 4 с лишним раза, все запросы уходят сразу и ждут ответ одновременно,
а не по очереди.

Потоки тоже ускоряют, но в пуле было 8 воркеров на 15 городов, то есть два захода. С 15 потоками вышло бы
примерно как у async, только дороже по ресурсам: 15 потоков ОС против одного event loop. На 15 запросах
это неважно, на тысячах уже да

Итого: в приложении sync, для массовых запросов async. Оба варианта есть в utils и переключаются в интерфейсе.